In [1]:
import gzip
import pickle

with gzip.open('dataset/train_set.pkl', 'rb') as file:
    train_set = pickle.load(file)
with gzip.open('dataset/test_set.pkl', 'rb') as file:
    test_set = pickle.load(file)
    
# Calculate the weight for each class in the training set
# Count the number of samples in each class in the label top20
class_counts = train_set['top20'].value_counts()
class_weights = {0: class_counts[1] / class_counts[0], 1: class_counts[0] / class_counts[1]}
print("Class weights: ", class_weights)


Class weights:  {0: 0.16575509770603228, 1: 6.032996956591382}


In [ ]:
import os
import time
import gzip
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
matplotlib.use('Agg')  # Disabilita il display interattivo
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

import Classifier  # Assicurati di avere la classe Classifier come da tuo codice
# Se Classifier non è un modulo installato, puoi sostituire con la definizione della rete,
# ad esempio:
# class Classifier(nn.Module):
#     def __init__(self, layers, structureNet, activation):
#         super(Classifier, self).__init__()
#         ...
#         # definisci la tua rete

BATCH_SIZE = 15000
NUM_EPOCHS = 10

def main():
    print("GRID SEARCH")
    print("Loading data...")

    # Caricamento dataset
    with gzip.open('dataset/train_set.pkl', 'rb') as file:
        train_set = pickle.load(file)
    with gzip.open('dataset/test_set.pkl', 'rb') as file:
        test_set = pickle.load(file)

    print("Data loaded!")

    seed = int(time.time() % 150)
    print("Seed: ", seed)

    # Preprocessing base
    print("Dataset processing...")
    train_set = train_set.dropna()
    test_set = test_set.dropna()

    # Features su cui addestrare
    features = ['performance_wma', 'top20_consistency', 'races_dby_consistency']
    print("Features: ", features)

    # Suddivisione X, y
    x_train = train_set[features]
    y_train = train_set['top20']
    x_test = test_set[features]
    y_test = test_set['top20']

    print("Dataset processed!")
    print("Dataset shape: ", x_train.shape, y_train.shape, x_test.shape, y_test.shape)

    input("Press Enter to continue...")  # Facoltativo

    # Seleziona device (GPU se disponibile)
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    print("Device: ", device)

    # Conversione in tensor
    print("Converting to tensor...")
    x_train_t = torch.tensor(x_train.values, dtype=torch.float32).to(device)
    y_train_t = torch.tensor(y_train.values, dtype=torch.float32).to(device)
    x_test_t = torch.tensor(x_test.values, dtype=torch.float32).to(device)
    y_test_t = torch.tensor(y_test.values, dtype=torch.float32).to(device)
    print("Converted to tensor!")

    # Configurazioni di rete da testare
    network_configs = [
        {
            "layers": [3, 8, 10, 15, 1],
            "activation": "tanh",
            "optimizer": "sgd",
            "penality": 0.0002,
            "momentum": 0.9,
            "learning_rate": 0.1
        },
        # Add others configurations
    ]

    numberTest = len(network_configs)
    
    bestResults = []

    # Dataloader (shuffle True solo per train)
    print("Creating data loader...")
    data_loader_train = torch.utils.data.DataLoader(
        list(zip(x_train_t, y_train_t)),
        batch_size=BATCH_SIZE,
        shuffle=True
    )
    data_loader_test = torch.utils.data.DataLoader(
        list(zip(x_test_t, y_test_t)),
        batch_size=BATCH_SIZE,
        shuffle=False
    )
    print("Data loader created!")

    print("Start training...")

    for number, config in enumerate(network_configs):
        print(f"\nTest[{number+1}/{numberTest}] --> Start training...")

        layers = config['layers']
        activation = config['activation']
        optimizerName = config['optimizer']
        penality = config['penality']
        momentum = config['momentum']
        lr = config['learning_rate']

        print(f"Test[{number+1}/{numberTest}] --> Layers: {layers}, Activation: {activation}, "
              f"Optimizer: {optimizerName}, Penality: {penality}, Momentum: {momentum}, Learning-rate: {lr}")

        # Name of the folder
        testName = f"{layers}-{optimizerName}-{activation}-{penality}-{momentum}-{lr}"
        pathName = f'models/Cup-{testName}'
        bestResults.append(testName + "\n")

        # Create directory
        print(f"Test[{number+1}/{numberTest}] --> Creating directory: {pathName}...")
        os.makedirs(pathName, exist_ok=True)
        print(f"Test[{number+1}/{numberTest}] --> Directory created!")

        # Create net
        print(f"Test[{number+1}/{numberTest}] --> Creating net...")
        structureNet = []  # Se ti serve in Classifier
        net = Classifier.Classifier(layers, structureNet, activation)
        net = net.to(device)
        net = net.float()
        print(f"Test[{number+1}/{numberTest}] --> Net created!")

        # OPTIMIZER
        criterion = nn.BCEWithLogitsLoss()
        if optimizerName == "adam":
            optimizer = optim.Adam(net.parameters(), lr=lr, weight_decay=penality)
        elif optimizerName == "sgd":
            optimizer = optim.SGD(net.parameters(), lr=lr, momentum=momentum, weight_decay=penality)
        else:
            print("OPTIMIZER NOT FOUND!")
            exit(1)

        net.train()
        loss_values_train = []

        # TRAIN LOOP
        for epoch in range(NUM_EPOCHS):
            total_loss = 0.0
            for batch_input, batch_output in data_loader_train:
                # Forward
                outputs = net(batch_input)
                loss = criterion(outputs, batch_output.view(-1, 1))
                total_loss += loss.item()

                # Backward
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            avg_loss_train = total_loss / len(data_loader_train)
            loss_values_train.append(avg_loss_train)
            print(f"Test[{number+1}/{numberTest}] --> "
                  f"Epoch[{epoch+1}/{NUM_EPOCHS}] --> Loss: {avg_loss_train:.6f}")

        print(f"Test[{number+1}/{numberTest}] --> Training finished!")

        # ----------------------------------------------------
        #  INFERENCE (prediction) WITHOUT SHUFFLE
        # ----------------------------------------------------
        net.eval()
        with torch.no_grad():
            pred_probs_train = torch.sigmoid(net(x_train_t)).cpu().numpy().ravel()
            pred_probs_test  = torch.sigmoid(net(x_test_t)).cpu().numpy().ravel()

        # ----------------------------------------------------
        #  REBUILD DATAFRAME CON PREDICTED_TOP20
        #  AND CALCULATE CONFUSION MATRIX AND CLASSIFICATION REPORT
        # ----------------------------------------------------
        train_df = train_set.copy()  # DataFrame con stesse righe/ordine di x_train
        train_df['pred_prob'] = pred_probs_train

        test_df = test_set.copy()    # DataFrame con stesse righe/ordine di x_test
        test_df['pred_prob'] = pred_probs_test

        # Funzione per impostare predicted_top20=1 ai primi 20 per race_id
        def evaluate_top_20(df, race_id_col='race_id', prob_col='pred_prob'):
            """
            Return DataFrame with column 'predicted_top20' set to
            1 for the top 20 rows of each group, 0 otherwise.
            """
            grouped_dfs = []
            for race_id, group in df.groupby(race_id_col):
                group = group.sort_values(by=prob_col, ascending=False).copy()
                group['predicted_top20'] = 0
                top_n = min(20, len(group))
                group.iloc[:top_n, group.columns.get_loc('predicted_top20')] = 1
                grouped_dfs.append(group)
            return pd.concat(grouped_dfs)

        # Applico la funzione
        train_df = evaluate_top_20(train_df)
        test_df  = evaluate_top_20(test_df)

        # ----------------------------------------------------
        #  CALCULATE CONFUSION MATRIX AND CLASSIFICATION REPORT
        # ----------------------------------------------------
        y_true_train = train_df['top20'].values
        y_pred_train = train_df['predicted_top20'].values

        y_true_test  = test_df['top20'].values
        y_pred_test  = test_df['predicted_top20'].values

        cm_train = confusion_matrix(y_true_train, y_pred_train)
        cr_train = classification_report(y_true_train, y_pred_train)
        cm_test = confusion_matrix(y_true_test, y_pred_test)
        cr_test = classification_report(y_true_test, y_pred_test)

        print("\nConfusion Matrix - Train\n", cm_train)
        print("Classification Report - Train\n", cr_train)

        print("Confusion Matrix - Test\n", cm_test)
        print("Classification Report - Test\n", cr_test)

        # Salvataggio delle confusion matrix su file PNG
        plt.figure(figsize=(10, 7))
        plt.title('Confusion Matrix - Train')
        sns.heatmap(cm_train, annot=True, fmt='d', cmap='Blues')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.savefig(f'{pathName}/Confusion-Matrix-Train.png')
        plt.clf()

        plt.figure(figsize=(10, 7))
        plt.title('Confusion Matrix - Test')
        sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.savefig(f'{pathName}/Confusion-Matrix-Test.png')
        plt.clf()

        # Salvataggio dei classification report su file di testo
        with open(f"{pathName}/scores-train.txt", "w") as file:
            file.write(cr_train)

        with open(f"{pathName}/scores-test.txt", "w") as file:
            file.write(cr_test)

        print(f"Test[{number+1}/{numberTest}] --> End training!\n")

if __name__ == "__main__":
    main()

GRID SEARCH
Loading data...
Data loaded!
Seed:  139
Dataset processing...
Features:  ['performance_wma', 'top20_consistency', 'races_dby_consistency']
Dataset processed!
Dataset shape:  (426364, 3) (426364,) (34509, 3) (34509,)
Device:  cuda:0
Converting to tensor...
Converted to tensor!
Creating data loader...
Data loader created!
Start training...

Test[1/1] --> Start training...
Test[1/1] --> Layers: [3, 8, 10, 15, 1], Activation: tanh, Optimizer: sgd, Penality: 0.0002, Momentum: 0.9, Learning-rate: 0.1
Test[1/1] --> Creating directory: models/Cup-[3, 8, 10, 15, 1]-sgd-tanh-0.0002-0.9-0.1...
Test[1/1] --> Directory created!
Test[1/1] --> Creating net...
Test[1/1] --> Net created!
Test[1/1] --> Epoch[1/10] --> Loss: 1.102787
Test[1/1] --> Epoch[2/10] --> Loss: 1.079904
Test[1/1] --> Epoch[3/10] --> Loss: 1.074659
Test[1/1] --> Epoch[4/10] --> Loss: 1.074287
Test[1/1] --> Epoch[5/10] --> Loss: 1.072676
Test[1/1] --> Epoch[6/10] --> Loss: 1.078532
Test[1/1] --> Epoch[7/10] --> Loss: 1.